In [1]:
import requests
import logging
import json
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, BaseMessage,
    ChatMessage, HumanMessageChunk, SystemMessageChunk, AIMessageChunk
)
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult
from typing import List, Dict, Any, Optional, Iterator
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

for key in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY', 'all_proxy', 'ALL_PROXY']:
    os.environ.pop(key, None)


In [3]:
class VLLMChatModel(BaseChatModel):
    base_url: str = "http://10.16.86.206:8000/v1"
    model_name: str = "Qwen/Qwen2.5-7B"
    api_key: str = "token-abc123"
    temperature: float = 0.4
    max_tokens: int = 1000
    repetition_penalty: float = 1.1
    top_p: float = 0.9
    stop: Optional[List[str]] = None

    model_config = {"extra": "allow"}

    def _normalize_messages(self, messages: List[BaseMessage]) -> List[BaseMessage]:
        normalized = []
        for msg in messages:
            if isinstance(msg, (HumanMessageChunk, SystemMessageChunk)):
                normalized.append(
                    HumanMessage(content=msg.content) if isinstance(msg, HumanMessageChunk)
                    else SystemMessage(content=msg.content)
                )
            elif isinstance(msg, ChatMessage):
                if msg.role == "system":
                    normalized.append(SystemMessage(content=msg.content))
                elif msg.role == "user":
                    normalized.append(HumanMessage(content=msg.content))
            elif isinstance(msg, (SystemMessage, HumanMessage)):
                normalized.append(msg)
        return normalized

    def _convert_to_api_messages(self, messages: List[BaseMessage]) -> List[Dict[str, str]]:
        api_messages = []
        for m in messages:
            role_map = {"system": "system", "human": "user", "ai": "assistant"}
            api_role = role_map.get(m.type, "user")
            api_messages.append({"role": api_role, "content": m.content})
        return api_messages

    def _prepare_payload(self, messages: List[BaseMessage], stop: Optional[List[str]] = None, stream: bool = False) -> Dict[str, Any]:
        normalized_msgs = self._normalize_messages(messages)
        logger.info(f"标准化后的消息列表：{[(type(m).__name__, m.content) for m in normalized_msgs]}")
        api_messages = self._convert_to_api_messages(normalized_msgs)
        logger.info(f"发送到 vLLM 的消息体：{api_messages}")

        payload = {
            "model": self.model_name,
            "messages": api_messages,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
            "top_p": self.top_p,
            "stop": stop or self.stop or ["<|im_end|>"],
            "repetition_penalty": self.repetition_penalty,
            "stream": stream
        }
        return payload

    def _stream(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager=None,
        **kwargs: Any
    ) -> Iterator[ChatGenerationChunk]:
        payload = self._prepare_payload(messages, stop, stream=True)
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}"
        }

        try:
            logger.info(f"请求 vLLM 流式 API：{self.base_url}/chat/completions")
            response = requests.post(
                url=f"{self.base_url}/chat/completions",
                json=payload,
                headers=headers,
                stream=True,
                timeout=60
            )
            response.raise_for_status()
            logger.info(f"响应状态码：{response.status_code}")

            for line in response.iter_lines(decode_unicode=True):
                if not line:
                    continue
                if line.startswith("data:"):
                    data_str = line[5:].strip()
                    if data_str == "[DONE]":
                        break
                    try:
                        chunk_data = json.loads(data_str)
                        choices = chunk_data.get("choices", [])
                        if choices:
                            delta = choices[0].get("delta", {})
                            content = delta.get("content")
                            finish_reason = choices[0].get("finish_reason")

                            if content is not None:
                                message_chunk = AIMessageChunk(content=content)
                                generation_chunk = ChatGenerationChunk(message=message_chunk)
                                yield generation_chunk

                            if finish_reason is not None:
                                logger.info(f"收到结束原因：{finish_reason}")
                                break
                    except json.JSONDecodeError as e:
                        logger.warning(f"解析 JSON 失败：{data_str}, 错误：{e}")
                        continue
        except Exception as e:
            raise RuntimeError(f"vLLM 流式 API 调用失败：{str(e)}") from e

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager=None,
        **kwargs: Any
    ) -> ChatResult:
        full_content = ""
        for chunk in self._stream(messages, stop, run_manager,** kwargs):
            full_content += chunk.message.content

        ai_message = AIMessage(content=full_content)
        generation = ChatGeneration(message=ai_message)
        return ChatResult(generations=[generation])

    @property
    def _llm_type(self) -> str:
        return "vllm-chat-model"

    @property
    def _identifying_params(self) -> Dict[str, Any]:
        return {
            "base_url": self.base_url,
            "model_name": self.model_name,
            "repetition_penalty": self.repetition_penalty
        }


In [ ]:
if __name__ == "__main__":
    llm = VLLMChatModel(
        base_url="http://10.16.86.206:8000/v1",
        model_name="Qwen/Qwen2.5-7B",
        repetition_penalty=1.1,
        max_tokens=1000
    )

    messages = [
        SystemMessage(content="You are a helpful assistant, answer in Chinese."),
        HumanMessage(content="重复说三遍“vLLM 高效部署大模型”，然后解释为什么 vLLM 并发高")
    ]

    print("===== 流式输出 =====")
    # ✅ 修复这里：stream返回AIMessageChunk，直接用chunk.content
    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n")

    print("===== 非流式输出 =====")
    response = llm.invoke(messages)
    print(response.content)

INFO:__main__:标准化后的消息列表：[('SystemMessage', 'You are a helpful assistant, answer in Chinese.'), ('HumanMessage', '重复说三遍“vLLM 高效部署大模型”，然后解释为什么 vLLM 并发高')]
INFO:__main__:发送到 vLLM 的消息体：[{'role': 'system', 'content': 'You are a helpful assistant, answer in Chinese.'}, {'role': 'user', 'content': '重复说三遍“vLLM 高效部署大模型”，然后解释为什么 vLLM 并发高'}]
INFO:__main__:请求 vLLM 流式 API：http://10.16.86.206:8000/v1/chat/completions
INFO:__main__:响应状态码：200


===== 流式输出 =====
好的，我将按照您的要求进行操作。

首先，让我们重复三次 "vLLM 高效部署大模型"：

1. vLLM 高效部署大模型。
2. vLLM 高效部署大模型。
3. vLLM 高效部署大模型。

现在，我来解释一下为什么 vLLM 的并发性能如此之高。

vLLM 是一种高效的大语言模型（Large Language Model）部署框架。它采用了多种优化技术，以提高在多核 CPU 和 GPU 上的并发处理能力。以下是一些关键因素：

1. **并行计算**：vLLM 利用现代处理器的多个核心和线程，并行执行任务，从而显著提高了计算效率。
2. **内存管理**：通过有效的内存管理和缓存策略，vLLM 能够减少数据访问延迟，进一步提升性能。
3. **算法优化**：使用更高效的算法和数据结构，例如分块矩阵乘法等，可以降低计算复杂度，加快推理速度。
4. **硬件加速支持**：vLLM 支持 CUDA 和 TensorRT 等硬件加速库，能够在 NVIDIA GPU 上实现高速推理。
5. **动态调度**：根据实际负载情况，vLLM 可以灵活调整任务分配和资源利用，确保系统始终处于最佳状态。

综上所述，vLLM 之所以具有极高的并发性能，是因为其结合了先进的软件设计、高效的硬件利用以及对各种应用场景的良好适应性。这些特性使得 vLLM 成为一个强大且实用的工具，适用于需要快速响应和大规模吞吐量的各种自然语言处理任务。

INFO:__main__:收到结束原因：stop
INFO:__main__:标准化后的消息列表：[('SystemMessage', 'You are a helpful assistant, answer in Chinese.'), ('HumanMessage', '重复说三遍“vLLM 高效部署大模型”，然后解释为什么 vLLM 并发高')]
INFO:__main__:发送到 vLLM 的消息体：[{'role': 'system', 'content': 'You are a helpful assistant, answer in Chinese.'}, {'role': 'user', 'content': '重复说三遍“vLLM 高效部署大模型”，然后解释为什么 vLLM 并发高'}]
INFO:__main__:请求 vLLM 流式 API：http://10.16.86.206:8000/v1/chat/completions
INFO:__main__:响应状态码：200




===== 非流式输出 =====


INFO:__main__:收到结束原因：stop


vLLM 高效部署大模型，vLLM 高效部署大模型，vLLM 高效部署大模型。vLLM 是一种高效的开源大语言模型推理库，它通过利用多线程和异步 I/O 技术来实现高性能的并发处理能力。这种设计使得 vLLM 能够在单个服务器上支持更多的用户请求，并且能够更快速地响应用户的查询。

具体来说，vLLM 采用了以下几种技术来提高并发性能：

1. 多线程：vLLM 利用 Python 的 `concurrent.futures` 模块中的 ThreadPoolExecutor 来管理多个工作线程。这些线程可以并行执行任务，从而加快了整体的计算速度。
2. 异步 I/O：为了进一步提升效率，vLLM 使用了 asyncio 库来进行异步 I/O 操作。这样，在等待数据读取或写入时，程序不会被阻塞，而是继续执行其他任务。
3. 数据流式传输：当生成文本时，vLLM 支持分段返回结果，而不是一次性将整个答案发送给客户端。这不仅减少了内存占用，还允许用户实时查看生成的内容。
4. 内存优化：vLLM 在设计中考虑到了内存使用问题，尽量减少不必要的内存消耗。例如，它会根据需要动态调整缓存大小以适应不同规模的任务。
5. 硬件加速：如果可用的话，vLLM 还能借助 GPU 或 TPU 等硬件设备进行加速运算，进一步提升了模型推理的速度。

综上所述，通过上述技术和方法的应用，vLLM 实现了较高的并发处理能力和优秀的性能表现，使其成为部署大规模语言模型的理想选择之一。
